# Homework 3 — Recursion and Recurrences

This notebook checks recursive Fibonacci call counts, compares recursive and iterative implementations, and analyzes several recurrences.

## 1. Counted recursive Fibonacci

`fib_counted(n)` returns both the Fibonacci value and the number of function calls in its recursion tree. As in the lecture version, each base-case call counts as one call. The proposed identity is `C(n) = 2 F(n+1) - 1`.

In [ ]:
def fib_counted(n):
    if n <= 1:
        return n, 1
    value_left, calls_left = fib_counted(n - 1)
    value_right, calls_right = fib_counted(n - 2)
    return value_left + value_right, calls_left + calls_right + 1


def fibonacci_value(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


tested_n = [10, 20, 25, 30, 32, 35]
measured_counts = {}
print(f"{'n':>3} {'F(n)':>12} {'measured calls':>18} {'2*F(n+1)-1':>14} {'match':>8}")
print('-' * 61)
for n in tested_n:
    value, calls = fib_counted(n)
    predicted_calls = 2 * fibonacci_value(n + 1) - 1
    measured_counts[n] = calls
    assert calls == predicted_calls
    print(f"{n:>3} {value:>12} {calls:>18,} {predicted_calls:>14,} {'yes':>8}")

print("\nAll measured call counts match 2*F(n+1)-1.")

The equality follows from the call-count recurrence: `C(0)=C(1)=1` and `C(n)=C(n-1)+C(n-2)+1`. If `C(n-1)=2F(n)-1` and `C(n-2)=2F(n-1)-1`, then `C(n)=2(F(n)+F(n-1))-1=2F(n+1)-1`. The assertions above check the identity empirically for the requested inputs.

## 2. Consecutive call-count ratios

For each `n` from 21 through 30, compute `C(n)/C(n-1)` and compare it with the golden ratio `φ = (1 + √5)/2`.

In [ ]:
import math

phi = (1 + math.sqrt(5)) / 2
counts_20_to_30 = {n: fib_counted(n)[1] for n in range(20, 31)}
print(f"phi = {phi:.12f}\n")
print(f"{'n':>3} {'C(n)':>14} {'C(n)/C(n-1)':>16} {'difference from phi':>21}")
print('-' * 59)
for n in range(21, 31):
    ratio = counts_20_to_30[n] / counts_20_to_30[n - 1]
    print(f"{n:>3} {counts_20_to_30[n]:>14,} {ratio:>16.10f} {abs(ratio - phi):>21.10f}")

The ratios approach `φ` (about 1.618034). Fibonacci numbers have consecutive-term ratio tending to `φ`; since `C(n)=2F(n+1)-1`, the constant factor and subtractive 1 do not change the limiting ratio.

## 3. Iterative Fibonacci and timing

The iterative implementation uses one loop and two state variables. For a fair timing comparison, the recursive version below returns only the Fibonacci value (it does not count calls, since counting adds overhead). Repeated measurements are used because timings depend on the machine and runtime.

In [ ]:
import statistics
import time


def fib_recursive(n):
    if n <= 1:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)


def fib_iterative(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


n_timing = 30
assert fib_recursive(n_timing) == fib_iterative(n_timing) == 832040
fib_recursive(n_timing)  # warm-up
fib_iterative(n_timing)  # warm-up

recursive_samples = []
for _ in range(5):
    start = time.perf_counter()
    fib_recursive(n_timing)
    recursive_samples.append(time.perf_counter() - start)

iterative_batch = 20_000
iterative_samples = []
for _ in range(5):
    start = time.perf_counter()
    for _ in range(iterative_batch):
        fib_iterative(n_timing)
    iterative_samples.append((time.perf_counter() - start) / iterative_batch)

recursive_seconds = statistics.median(recursive_samples)
iterative_seconds = statistics.median(iterative_samples)
print(f"n = {n_timing}")
print(f"Recursive median: {recursive_seconds:.6f} seconds per call")
print(f"Iterative median: {iterative_seconds:.9f} seconds per call")
print(f"Recursive / iterative timing ratio: {recursive_seconds / iterative_seconds:,.1f}x")

The naive recursive implementation takes `Θ(φⁿ)` time (often bounded as `O(2ⁿ)`) because it repeats subproblems; the iterative implementation takes `Θ(n)` time and `Θ(1)` extra space. The timing ratio is an empirical result for this runtime, not a machine-independent constant.

## 4. Recurrences and Master Theorem parameters

For `T(n) = aT(n/b) + f(n)`, `a` is the number of subproblems, `n/b` is each subproblem's size, and `f(n)` is the non-recursive work. The `rev` answer assumes Exercise 3 reverses a sequence recursively by reversing its tail and appending the first element; its precise source was not included in the provided workspace.

| Algorithm | Recurrence | `a`, `b`, `f(n)` | Asymptotic solution |
|---|---|---|---|
| Binary search | `T(n) = T(n/2) + Θ(1)` | `a=1`, `b=2`, `f(n)=Θ(1)` | `Θ(log n)` |
| Merge sort `divide_shape` (two halves, linear combine) | `T(n) = 2T(n/2) + Θ(n)` | `a=2`, `b=2`, `f(n)=Θ(n)` | `Θ(n log n)` |
| `rev` (assumed tail recursion plus append) | `T(n) = T(n-1) + Θ(n)` | Not in Master form (`n-1`, not `n/b`) | `Θ(n²)` |
| Three calls on `n/4`, with quadratic glue | `T(n) = 3T(n/4) + Θ(n²)` | `a=3`, `b=4`, `f(n)=Θ(n²)` | `Θ(n²)` |

For the last recurrence, `n^(log₄ 3)` grows more slowly than `n²`, so the quadratic glue dominates (Master Theorem case 3). For `rev`, summing the linear work across input sizes gives `n + (n-1) + … + 1 = Θ(n²)`. If Exercise 3 defines `rev` differently (for example, with an accumulator), its recurrence may differ.